# 02 — Modeling (iterasi 4)

```
Pipeline (dipakai untuk LightGBM dan XGBoost)
├── struct : StructuralFeatures          kode wilayah, rumah tangga, proksi fisika
├── btype  : BuildingTypeClusterer       (opsional) KMeans tipe bangunan
├── vtype  : VulnerabilityTyper          (opsional) tipe kerentanan supervised
├── prep   : ColumnTransformer
│   ├── num | cat | grp | te | nb
│   ├── hh  : HouseholdEncoder           (opsional) bangunan serumah
│   ├── adj : AdjacentAreaEncoder        (opsional) ward / blok tetangga
│   └── ri  : ResidualIntensityEncoder   (BARU) intensitas guncangan bersih, dua tahap
└── clf    : LGBMClassifier | XGBGradeClassifier
                ↓
BlendClassifier: α·LightGBM + (1−α)·XGBoost  →  bobot kelas untuk macro F1
```

### Yang baru di iterasi 4
| Perubahan | Alasan |
|---|---|
| **`ResidualIntensityEncoder`** | Target encoding wilayah biasa mencampur *intensitas guncangan* dengan *komposisi bangunan*. Tahap 1 memprediksi grade dari konstruksi saja; tahap 2 merata-ratakan sisa (grade − prediksi) per wilayah dan per tetangga sehingga tersisa intensitas guncangan bersih |
| **Blend LightGBM + XGBoost** resmi di pipeline | blend di iterasi 3 menaikkan public LB 0,60228 → 0,60324 |
| **LightGBM lebih teliti** (lr 0,05, 900 pohon) | learning rate kecil + pohon lebih banyak biasanya sedikit lebih akurat |
| **Ablation memakai LightGBM cepat** | hanya *selisih* antar set fitur yang penting saat ablation |
| **Smoke test** di awal | semua jalur kode dijalankan pada 3% data (±3–5 menit) sebelum run panjang |
| Output di `/kaggle/working/` (di luar repo) | clone ulang repo tidak lagi menghapus hasil |

**Estimasi waktu CPU Kaggle:** smoke test ±5 menit · ablation ±40 menit · CV LightGBM ±60 menit ·
CV XGBoost ±60 menit · model final ±25 menit. Jalankan dengan **Save Version → Save & Run All**.

In [ ]:
# === Setup: lokasi repo (Kaggle: selalu clone versi terbaru dari GitHub) ===
import sys, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/yafifaldin/structural-damage-classification.git"
ON_KAGGLE = Path("/kaggle/working").exists()

ROOT = None
if ON_KAGGLE:
    ROOT = Path("/kaggle/working/repo")
    shutil.rmtree(ROOT, ignore_errors=True)
    try:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
        print(subprocess.run(["git", "-C", str(ROOT), "log", "-1", "--oneline"],
                             capture_output=True, text=True).stdout)
    except Exception as exc:  # tanpa Internet: cari repo yang di-upload sebagai Kaggle Dataset
        print("git clone gagal (Internet off?):", exc)
        hits = sorted(Path("/kaggle/input").glob("**/scripts/__init__.py"))
        ROOT = hits[0].parent.parent if hits else None
else:
    for cand in (Path.cwd(), Path.cwd().parent):
        if (cand / "scripts" / "__init__.py").exists():
            ROOT = cand.resolve()
            break
assert ROOT is not None, "Folder repo (berisi scripts/) tidak ditemukan."

for name in [m for m in sys.modules if m == "scripts" or m.startswith("scripts.")]:
    del sys.modules[name]          # pastikan modul versi terbaru yang dipakai
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

In [ ]:
import json, time
import numpy as np
import pandas as pd
from IPython.display import display, Image

from scripts.preprocess import load_data, split_xy, output_dir, resolve_config, DEFAULT_CONFIG
from scripts.predict import predict_grades, format_submission, save_submission
from scripts import train as T

# ---- konfigurasi ----
SMOKE_TEST = True      # cek cepat semua jalur kode dulu (disarankan)
ABLATION = True        # pilih konfigurasi fitur otomatis
ABLATION_FRAC = 0.30
MODELS = ("lgbm", "xgb")
N_SPLITS = 5
CONFIG = dict(DEFAULT_CONFIG)   # dipakai bila ABLATION = False

OUT = output_dir()
train, test, sample = load_data()
X, y = split_xy(train)
print("output dir:", OUT, "| X:", X.shape)

## 0. Smoke test

In [ ]:
if SMOKE_TEST:
    print(T.smoke_test(X, y, test))

## 1. Ablation fitur

In [ ]:
if ABLATION:
    t0 = time.time()
    CONFIG, ablation = T.run_ablation(X, y, frac=ABLATION_FRAC)
    ablation.to_csv(OUT / "reports" / "ablation_results.csv", index=False)
    print(f"ablation selesai dalam {(time.time() - t0) / 60:.1f} menit")
    display(ablation)
CONFIG = resolve_config(CONFIG)
print("konfigurasi terpilih:", CONFIG)
T.build_pipeline(config=CONFIG)

## 2. Cross-validation 5-fold per model (fold identik)

In [ ]:
oofs = {}
for m in MODELS:
    t0 = time.time()
    oofs[m], fold_ids, classes = T.cross_validate_oof(X, y, None, CONFIG, n_splits=N_SPLITS, model=m)
    T.save_oof(oofs[m], fold_ids, y, OUT / "reports" / f"oof_{m}.npz")
    print(f"CV {m} selesai dalam {(time.time() - t0) / 60:.1f} menit")
y_idx = np.searchsorted(classes, y)

## 3. Blend + bobot kelas untuk macro F1

In [ ]:
if len(MODELS) == 2:
    alpha, blend_table, heldout = T.optimize_blend(oofs[MODELS[0]], oofs[MODELS[1]], y_idx, fold_ids)
    blend_weights = [alpha, 1 - alpha]
    blend_table.to_csv(OUT / "reports" / "blend_search.csv", index=False)
    blended = alpha * oofs[MODELS[0]] + (1 - alpha) * oofs[MODELS[1]]
    print(f"alpha {MODELS[0]} = {alpha}")
    print(pd.Series({MODELS[0]: heldout["a"], MODELS[1]: heldout["b"], "blend": heldout["blend"]}).round(4))
else:
    blend_weights, blended, heldout = [1.0], oofs[MODELS[0]], {}

cv = T.summarize_cv(blended, y, fold_ids, classes)
print("macro F1 held-out per fold:", cv["macro_f1_heldout_per_fold"], "| mean:", cv["macro_f1_heldout_mean"])
print("bobot kelas:", cv["class_weights"])
display(pd.DataFrame(cv["report"]).T.round(4))
T.plot_confusion(y, classes[(blended * np.asarray(cv["class_weights"])).argmax(1)])
Image(str(OUT / "reports" / "figures" / "confusion_matrix.png"))

## 4. Model final (blend, seluruh data train) + simpan

In [ ]:
t0 = time.time()
model = T.fit_blend_model(X, y, CONFIG, None, blend_weights, cv["class_weights"], MODELS)
model_path = T.save_model(model)
print(f"model final: {(time.time() - t0) / 60:.1f} menit -> {model_path}")
imp = T.plot_feature_importance(model)
Image(str(OUT / "reports" / "figures" / "feature_importance.png"))

In [ ]:
summary = {"config": CONFIG, "models": list(MODELS),
           "blend_weights": dict(zip(MODELS, np.round(blend_weights, 2).tolist())),
           "heldout_by_model": {k: round(v, 4) for k, v in heldout.items()},
           **{k: v for k, v in cv.items() if k != "report"},
           "per_class_f1": {k: round(v["f1-score"], 4) for k, v in cv["report"].items()
                            if k not in ("accuracy", "macro avg", "weighted avg")},
           "top_features": imp.head(20).round(0).to_dict()}
with open(OUT / "reports" / "cv_summary.json", "w") as fh:
    json.dump(summary, fh, indent=2, default=str)
print("===== RINGKASAN ITERASI 4 (salin semua di bawah ini) =====")
if ABLATION:
    print(ablation[["step", "macro_f1_heldout", "gain", "kept"]].to_string(index=False))
print(json.dumps(summary, default=str))

## 5. Submission (label angka 1–5)

In [ ]:
sub = format_submission(sample, test["building_id"], predict_grades(model, test))
path = save_submission(sub)
print("submission:", path, "+ /kaggle/working/submission.csv", sub.shape)
print(sub.head())
sub["damage_grade"].value_counts(normalize=True).sort_index().round(3)